# Gaussian Drift OPT Debug

Clean propagation check for the homogeneous 2D acoustic OPT operator. The baseline here is `YorderBspace = -1`, `YorderBtime = -1`, because that means the direct/indicator factor in `integralWYYKK` and avoids the ambiguous `YorderB = 0` case.


## 1. Setup


In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()


include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT

include("temporaryHelpers.jl")\n

## 2. Controls


In [ ]:
shape = (201, 201)
center = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))

velocity_value = 2600.0
dx = 100.0
cfl = 0.45
dt = cfl * dx / velocity_value
delta = (dx, dx, dt)
velocity = fill(velocity_value, shape)

Nt = 180
store_every = 3
sigma = 10.0
amplitude = 1.0
init_gaussian = gaussian_field(shape, center; sigma=sigma, amplitude=amplitude)

pointsInSpace = 3
pointsInTime = 3
supplementaryOrder = 2
orderBspace = 1
orderBtime = 1
YorderBspace = -1
YorderBtime = -1

@show shape center velocity_value delta Nt store_every sigma
@show pointsInSpace pointsInTime supplementaryOrder orderBspace orderBtime YorderBspace YorderBtime


## 3. Unit Stencil Sanity Check


In [ ]:
toy_unit = build_toy_opt_prepared(
    famousEquationType="2DacousticHomoTime",
    velocity_value=1.0,
    shape=shape,
    dx=1.0,
    dt=1.0,
    pointsInSpace=pointsInSpace,
    pointsInTime=pointsInTime,
    supplementaryOrder=supplementaryOrder,
    orderBspace=orderBspace,
    orderBtime=orderBtime,
    YorderBspace=YorderBspace,
    YorderBtime=YorderBtime,
)

st_unit = operator_stencil_at_point(toy_unit.numOps, center; which=:left)
unit_matrices = stencil_matrices_by_time(st_unit)
unit_sums = sum.(getfield.(unit_matrices, :matrix))

stencil_time_summary(st_unit), unit_matrices, unit_sums, compare_stencil_scale_to_fd(st_unit, 1.0, 1.0, 1.0)


## 4. FD Baseline Propagation


In [ ]:
preparedFD = prepare_fd2d_acoustic_baseline(velocity, delta)
frames_fd = propagate_linear_frames_from_initial(
    preparedFD,
    init_gaussian,
    init_gaussian,
    Nt;
    store_every=store_every,
    blowup_limit=1e12,
)

fd_report = wavefield_snapshot_report(frames_fd)
fd_drift = drift_report(frames_fd, center)
fd_argmax = argmax_report(frames_fd, center)
fd_symmetry = symmetry_report(frames_fd, center)

@show length(frames_fd) fd_report[1] fd_report[end]
@show fd_drift[1] fd_drift[end]
@show fd_argmax[1] fd_argmax[end]
@show fd_symmetry[1] fd_symmetry[end]


In [ ]:
fig_fd = plot_wave_snapshots(
    frames_fd;
    sourcePoint=center,
    title="FD baseline: homogeneous Gaussian initial condition",
)
fig_fd


## 5. OPT Propagation With Y = -1


In [ ]:
toyOPT = build_toy_opt_prepared(
    famousEquationType="2DacousticHomoTime",
    velocity_value=velocity_value,
    shape=shape,
    dx=dx,
    cfl=cfl,
    pointsInSpace=pointsInSpace,
    pointsInTime=pointsInTime,
    supplementaryOrder=supplementaryOrder,
    orderBspace=orderBspace,
    orderBtime=orderBtime,
    YorderBspace=YorderBspace,
    YorderBtime=YorderBtime,
)
preparedOPT = toyOPT.prepared

opt_A_report = implicit_matrix_report(preparedOPT)
st_opt = operator_stencil_at_point(toyOPT.numOps, center; which=:left)
opt_matrices = stencil_matrices_by_time(st_opt)
opt_sums = sum.(getfield.(opt_matrices, :matrix))

@show preparedOPT.spaceShape preparedOPT.NpointsSpace preparedOPT.timePointsUsedForOneStep
@show size(preparedOPT.A_unknown) nnz(preparedOPT.A_unknown) nnz(preparedOPT.L_known)
@show opt_A_report
stencil_time_summary(st_opt), opt_matrices, opt_sums


In [ ]:
frames_opt = propagate_linear_frames_from_initial(
    preparedOPT,
    init_gaussian,
    init_gaussian,
    Nt;
    store_every=store_every,
    blowup_limit=1e12,
)

opt_report = wavefield_snapshot_report(frames_opt)
opt_drift = drift_report(frames_opt, center)
opt_argmax = argmax_report(frames_opt, center)
opt_symmetry = symmetry_report(frames_opt, center)

@show length(frames_opt) opt_report[1] opt_report[end]
@show opt_drift[1] opt_drift[end]
@show opt_argmax[1] opt_argmax[end]
@show opt_symmetry[1] opt_symmetry[end]


In [ ]:
fig_opt = plot_wave_snapshots(
    frames_opt;
    sourcePoint=center,
    title="OPT Y=-1: homogeneous Gaussian initial condition",
)
fig_opt


## 6. Comparison


In [ ]:
drift_comparison = (
    fd_final_drift = fd_drift[end],
    opt_final_drift = opt_drift[end],
    fd_final_argmax = fd_argmax[end],
    opt_final_argmax = opt_argmax[end],
    fd_final_symmetry = fd_symmetry[end],
    opt_final_symmetry = opt_symmetry[end],
    fd_final_max = maximum(abs, frames_fd[end]),
    opt_final_max = maximum(abs, frames_opt[end]),
)
drift_comparison


## 7. Optional Parameter Sweep


In [ ]:
parameter_cases = [
    (pointsInSpace=3, supplementaryOrder=2, YorderBspace=-1, YorderBtime=-1),
    (pointsInSpace=5, supplementaryOrder=2, YorderBspace=-1, YorderBtime=-1),
]

sweep_rows = NamedTuple[]
for case in parameter_cases
    println("building OPT case ", case)
    a_report = nothing
    try
        toy = build_toy_opt_prepared(
            famousEquationType="2DacousticHomoTime",
            velocity_value=velocity_value,
            shape=shape,
            dx=dx,
            cfl=cfl,
            pointsInSpace=case.pointsInSpace,
            pointsInTime=pointsInTime,
            supplementaryOrder=case.supplementaryOrder,
            orderBspace=orderBspace,
            orderBtime=orderBtime,
            YorderBspace=case.YorderBspace,
            YorderBtime=case.YorderBtime,
        )
        a_report = implicit_matrix_report(toy.prepared)
        if a_report.stored_nonfinite > 0 || a_report.zero_rows_finite > 0
            push!(sweep_rows, (; case..., status=:bad_A_unknown, matrix_report=a_report, nframes=0, final_drift=nothing, final_symmetry=nothing, final_max=NaN))
            @warn "Skipping OPT case because A_unknown is non-finite or has finite-zero rows" case matrix_report=a_report
            continue
        end
        fr = propagate_linear_frames_from_initial(toy.prepared, init_gaussian, init_gaussian, Nt; store_every=store_every, blowup_limit=1e12)
        dr = drift_report(fr, center)
        sy = symmetry_report(fr, center)
        push!(sweep_rows, (; case..., status=:ok, matrix_report=a_report, nframes=length(fr), final_drift=dr[end], final_symmetry=sy[end], final_max=maximum(abs, fr[end])))
    catch err
        push!(sweep_rows, (; case..., status=Symbol(nameof(typeof(err))), matrix_report=a_report, nframes=0, final_drift=nothing, final_symmetry=nothing, final_max=NaN))
        @warn "Skipping failed OPT case" case matrix_report=a_report exception=(err, catch_backtrace())
    end
end
sweep_rows
